# Wikidata tokens → semantic categories with MiniLM**What this notebook does.** It takes the 538 unique Wikidata type labels attached toToronto census-tract zones and groups them into a small set of functional categories(Transport, Education, Retail, …).**The method, in one line:** MiniLM measures how similar the labels are, hierarchicalclustering turns those similarities into candidate groups, and then a manual pass fixesthe groups that make no sense for an urban / cycling context.**Why MiniLM does not decide the categories on its own.** MiniLM sees short strings, notbuildings. It puts `metro_station`, `fire_station` and `coal-fired_power_station` in thesame group because they all end in "station". That is a real lexical similarity, but itis useless for predicting bike trips. So MiniLM proposes, the researcher disposes.**Run order.** Top to bottom. Sections 1–9 build the mapping. Section 10 tests whetherthe mapping is actually worth using — do not skip it.

---## 0. SetupInstall once, then restart the kernel if anything was newly installed.

In [ ]:
# Install the four things this notebook needs.# - sentence-transformers : the MiniLM encoder# - pyarrow              : lets pandas read .parquet files# - scikit-learn         : hierarchical clustering + silhouette score# %pip install sentence-transformers pyarrow scikit-learn pandas numpyimport numpy as npimport pandas as pdfrom sentence_transformers import SentenceTransformerfrom sklearn.cluster import AgglomerativeClusteringfrom sklearn.metrics import silhouette_score# Path to the melted token file: one row per (zone, wikidata_token) pair.DATA_PATH = "melted_wikidata_tokens.parquet"# Where to write results.OUT_DIR = "."# How many clusters to cut the hierarchy at.# This is deliberately MORE than the number of final categories we want.# Over-segmenting first and merging by hand afterwards is safer than# under-segmenting, because a cluster that mixes two unrelated things# is much harder to fix than two clusters that should be merged.K_CLUSTERS = 24pd.set_option("display.width", 200)pd.set_option("display.max_colwidth", 120)

---## 1. Load the dataThe file is "melted": one row per zone-token pair, so a zone with 5 Wikidata entitiesappears on 5 rows. Tokens repeat heavily across zones.

In [ ]:
raw = pd.read_parquet(DATA_PATH)print("rows          :", len(raw))print("unique zones  :", raw["loc_id"].nunique())print("unique tokens :", raw["wikidata_token"].nunique())print()print(raw.head())

In [ ]:
# How often does each token appear? This matters a lot later.freq = raw["wikidata_token"].value_counts()print("--- 15 most common tokens ---")print(freq.head(15).to_string())print()# The long tail is the real story: most tokens are almost never seen.print(f"tokens appearing exactly once : {(freq == 1).sum()}  "      f"({100 * (freq == 1).mean():.0f}% of all tokens)")print(f"tokens appearing 5+ times     : {(freq >= 5).sum()}  "      f"covering {100 * freq[freq >= 5].sum() / len(raw):.0f}% of rows")print()# Tokens per zone. The median is what limits everything downstream:# if a typical zone only has ~3 labels, no encoder can extract much from it.per_zone = raw.groupby("loc_id")["wikidata_token"].count()print(f"tokens per zone: median {per_zone.median():.0f}, "      f"mean {per_zone.mean():.1f}, max {per_zone.max()}")

---## 2. Clean the token textTwo reasons to clean before encoding:1. `metro_station` is not an English phrase. The tokeniser would split the underscore   awkwardly. `metro station` is what the model was trained on.2. We only need to embed **unique** tokens. `neighborhood` appears 148 times but its   embedding is identical every time, so encoding all 2,707 rows would be ~5× wasted work   for exactly the same numbers.Hyphens stay as they are — `coal-fired` and `drive-in` are genuine English hyphenation.Case is left alone because MiniLM is uncased; its tokeniser lowercases everything anyway.

In [ ]:
# Step 1: collapse to the unique token list. Sorting makes the run reproducible,# so cluster numbering does not shuffle between sessions.unique_tokens = (pd.Series(raw["wikidata_token"].dropna().unique())                   .sort_values()                   .reset_index(drop=True))# Turn a Wikidata token into a natural-language phrase.def clean(token: str) -> str:    s = token.replace("_", " ")   # metro_station -> metro station    s = " ".join(s.split())       # collapse any double spaces    return sclean_text = unique_tokens.map(clean)tokens = pd.DataFrame({"wikidata_token": unique_tokens, "clean_text": clean_text})print(f"unique tokens to encode: {len(tokens)}")print()print(tokens.sample(8, random_state=0).to_string(index=False))

---## 3. Generate the MiniLM embeddings`all-MiniLM-L6-v2` turns each phrase into a 384-dimensional vector. Phrases that meansimilar things land near each other.`normalize_embeddings=True` gives every vector length 1. That matters because it makesthe dot product equal to cosine similarity, which is what we compare with in the nextstep.

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")E = model.encode(    tokens["clean_text"].tolist(),    batch_size=64,    normalize_embeddings=True,   # unit length -> dot product == cosine similarity    show_progress_bar=True,).astype(np.float32)print("embedding matrix:", E.shape)   # (538, 384)np.save(f"{OUT_DIR}/minilm_token_embeddings.npy", E)

In [ ]:
# Sanity check before trusting anything downstream.# If these numbers look wrong, the model or the text cleaning is broken.lookup = {t: i for i, t in enumerate(tokens["wikidata_token"])}# Cosine similarity between two tokens (vectors are already unit length).def cos(a, b):    return float(E[lookup[a]] @ E[lookup[b]])checks = [    ("urban_park",    "park",                 "should be HIGH"),    ("metro_station", "railway_station",      "should be HIGH"),    ("high_school",   "primary_school",       "should be HIGH"),    ("metro_station", "bakery",               "should be LOW"),    ("cemetery",      "sushi-ya",             "should be LOW"),    # This one is the warning sign. Two completely different urban functions,    # but they share the word "station", so MiniLM scores them as related.    ("metro_station", "fire_station",         "should be LOW, but will not be"),]for a, b, note in checks:    print(f"cos({a:16s}, {b:18s}) = {cos(a, b):.3f}   <- {note}")

---## 4. Cosine similarity between all token pairsThe full 538 × 538 similarity matrix. We convert it to a **distance** matrix(`distance = 1 − similarity`) because clustering algorithms want distances.

In [ ]:
S = E @ E.T                 # cosine similarity, because vectors are unit lengthD = 1.0 - S                 # cosine distancenp.fill_diagonal(D, 0.0)    # a token is exactly 0 distance from itselfD = np.clip(D, 0, 2)        # guard against tiny negative values from float error# Look at the upper triangle only, so each pair is counted once.iu = np.triu_indices_from(S, 1)pairs = S[iu]print(f"mean similarity   : {pairs.mean():.3f}")print(f"median similarity : {np.median(pairs):.3f}")print(f"95th percentile   : {np.percentile(pairs, 95):.3f}")print()print(f"pairs above 0.90  : {(pairs > 0.90).sum()}")print(f"pairs above 0.80  : {(pairs > 0.80).sum()}")print(f"pairs above 0.70  : {(pairs > 0.70).sum()}")print()print("Read this as: most token pairs are unrelated (median ~0.2), and only a small")print("number are strongly related. That is expected and fine.")

---## 5. How many clusters?We sweep `k` and score each result with the **silhouette score** — a measure of howwell-separated the clusters are. Higher is better, roughly on a −1 to +1 scale.Three linkage rules are compared:- **average** — merges groups by mean distance. Tends to "chain": one giant cluster  swallows everything.- **complete** — merges by worst-case distance. Produces tighter but sometimes arbitrary  groups.- **ward** — minimises within-cluster variance. Produces the most balanced sizes. It  assumes Euclidean distance, which is fine here because on unit-length vectors Euclidean  distance and cosine distance rank pairs identically.Watch for whether the silhouette **peaks** at some k. If it just keeps rising, the datahas no natural number of clusters and you have to choose one yourself.

In [ ]:
rows = []for k in [6, 8, 10, 12, 14, 16, 18, 20, 24, 28, 32, 40]:    for linkage in ["average", "complete", "ward"]:        if linkage == "ward":            # ward needs the raw vectors, not a distance matrix            labels = AgglomerativeClustering(n_clusters=k, linkage="ward").fit_predict(E)        else:            labels = AgglomerativeClustering(                n_clusters=k, metric="precomputed", linkage=linkage).fit_predict(D)        sizes = np.bincount(labels)        rows.append({            "k": k,            "linkage": linkage,            "silhouette": round(silhouette_score(D, labels, metric="precomputed"), 3),            # If the largest cluster holds most of the tokens, the clustering has failed            # even if the silhouette looks acceptable.            "largest_cluster": int(sizes.max()),        })sweep = pd.DataFrame(rows).pivot(index="k", columns="linkage",                                 values=["silhouette", "largest_cluster"])print(sweep.to_string())

**What the sweep shows.** Silhouette scores are low everywhere (roughly 0.03–0.13) andclimb steadily with k instead of peaking. There is no elbow. Two conclusions:1. Short labels do not form tight, well-separated clusters. Expect messy groups.2. `k` is a **researcher decision**, not something the data determines.`average` linkage is unusable here — at k = 6 one cluster holds 484 of the 538 tokens.We use **ward**, which gives balanced sizes and the best silhouette at every k.

---## 6. Cluster at k = 24 and read every cluster24 is deliberately more than we need. We will merge groups by hand in section 8.Printing the members with their occurrence counts is the important part — this is theevidence you use to name and correct the clusters, and it is what you would show asupervisor or examiner.

In [ ]:
labels = AgglomerativeClustering(n_clusters=K_CLUSTERS, linkage="ward").fit_predict(E)tokens["minilm_cluster"] = labelstokens["n_occurrences"] = tokens["wikidata_token"].map(freq)for c in range(K_CLUSTERS):    sub = tokens[tokens["minilm_cluster"] == c].sort_values("n_occurrences",                                                            ascending=False)    print(f"=== cluster {c:02d} — {len(sub)} tokens, "          f"{sub['n_occurrences'].sum()} occurrences ===")    print("  " + ", ".join(f"{r.wikidata_token}({r.n_occurrences})"                           for r in sub.itertuples()))    print()

---## 7. Name the clustersEach cluster gets the name of whatever function dominates it.**A note on how the names are attached.** The obvious approach is a dictionary keyed onthe cluster number (`{0: "Parks", 1: "Food", ...}`). That breaks silently: change themodel version, the sklearn version, or the input slightly, and cluster 0 becomes adifferent set of tokens while keeping the label "Parks".So instead we key on an **anchor token**. "The cluster containing `park` is the parkscluster" survives renumbering. If an anchor goes missing or two anchors collide, the codetells you instead of quietly mislabelling a third of your data.

In [ ]:
# anchor token -> name for the cluster that contains it.# Anchors were chosen after reading the printout in section 6.ANCHORS = {    "park":                                "Parks, Nature & Recreation",    "restaurant":                          "Food, Drink & Hospitality",    "movie_theater":                       "Culture & Performing Arts",    "performing_arts_center":              "Culture & Performing Arts",    "university":                          "Education",    "high_school":                         "Education",    "cemetery":                            "Memorials & Cemeteries",    "neighborhood":                        "Settlement & Districts",    "community_center":                    "Government, Civic & Community",    "museum":                              "Museums, Heritage & Attractions",    "law_library":                         "Libraries & Archives",    "library":                             "Libraries & Archives",    "tram_stop":                           "Transport & Transit",    "metro_station":                       "Transport & Transit",    "church_building":                     "Religion & Worship",    "nonprofit_organization":              "Organisations & Institutions",    "event_venue":                         "Sport & Event Venues",    "federal_electoral_district_of_Canada": "Administrative & Electoral Units",    "skyscraper":                          "Generic Buildings & Structures",    "observation_tower":                   "Generic Buildings & Structures",    "bridge":                              "Roads, Paths & Bridges",    "shopping_center":                     "Retail & Commercial",    "tennis_tournament_edition":           "Events & Incidents",    # cluster 07 is the leftover bin: ~57 mostly one-off tokens with nothing in common.    # It gets an honest name rather than a pretend one.    "proposed_entity":                     "Other / Unclassified",}# Resolve each cluster number to a name by looking at which anchors landed in it.cluster_name = {}for c in range(K_CLUSTERS):    members = set(tokens.loc[tokens["minilm_cluster"] == c, "wikidata_token"])    hits = [name for tok, name in ANCHORS.items() if tok in members]    if not hits:        # No anchor fell here. Do not guess — flag it and read the cluster yourself.        cluster_name[c] = "Other / Unclassified"        print(f"WARNING cluster {c:02d}: no anchor token. Inspect it and add an anchor.")    elif len(set(hits)) > 1:        # Two anchors that should be in different clusters merged into one.        cluster_name[c] = hits[0]        print(f"WARNING cluster {c:02d}: conflicting anchors {sorted(set(hits))}. "              f"Using '{hits[0]}'.")    else:        cluster_name[c] = hits[0]tokens["cluster_name"] = tokens["minilm_cluster"].map(cluster_name)print()print("--- cluster -> name ---")audit = (tokens.groupby(["minilm_cluster", "cluster_name"])               .size().rename("n_tokens").reset_index())print(audit.to_string(index=False))

---## 8. Manual correctionThis is the step that makes the categories defensible. MiniLM grouped by wording; weregroup by urban function.The four biggest problems it created:| Cluster | What MiniLM did | Why it is wrong ||---|---|---|| the "station" cluster | `metro_station`, `fire_station`, `radio_station`, `coal-fired_power_station`, `filling_station` together | shared suffix, four different functions || the "church" cluster | `Catholic_school` (37 uses), `Jesuit_school`, `yeshiva` filed under religion | the religious adjective outweighed the word "school" || the "park" cluster | `street`, `road`, `alley`, `bikeway` filed under parks | grouped by being outdoors, not by function || the "university" cluster | `hospital` (32 uses) filed under education | pulled in by "academic institution" wording |Everything below is an explicit, auditable decision. Report the correction rate in yourmethods section — it is a finding about how well short-label embeddings work, not anembarrassment.

In [ ]:
OVERRIDE = {}# Force these tokens into this category, whatever the cluster said.def assign(category, *token_list):    for t in token_list:        OVERRIDE[t] = category# --- street network and active travel: cycling-relevant, so kept separate from parks ---assign("Roads, Paths & Bridges",       "street", "road", "alley", "bikeway", "radial_route", "bus_lane",       "pedestrian_walkway")# --- rail/transit items that landed elsewhere ---# "Spanish_solution" is a Wikidata term for a station platform layout, not a place type.assign("Transport & Transit",       "classification_yard", "flat_crossing", "Spanish_solution",       "Toronto_streetcar_loop")# --- the "-station" false friends, plus industrial land uses ---assign("Utilities & Industry",       "assembly_plant", "factory", "munitions_factory", "warehouse",       "warehouse_district", "port", "dam", "distribution_center", "infrastructure",       "coal-fired_power_station", "gas-fired_power_station",       "natural_gas-fired_power_station", "pumping_station", "television_tower")# --- retail. "underground_city" is the PATH network: a shopping concourse ---assign("Retail & Commercial",       "farmers'_market", "market_hall", "record_shop", "music_store",       "video_rental_shop", "bookstore", "LGBTQ+_bookshop", "filling_station",       "underground_city", "commercial_building", "bank_building")assign("Sport & Event Venues",       "fair_ground", "arena", "convention_center", "country_club", "yacht_club", "pitch")# --- green space and public open space. Golf courses are large green land uses ---assign("Parks, Nature & Recreation",       "golf_course", "golf_club", "bathhouse", "bingo_hall", "leisure_center",       "stream", "square", "privately_owned_public_space", "public_space",       "lifeguard_tower")# --- public services, emergency services, government ---assign("Government, Civic & Community",       "prison", "hackspace", "city_hall", "drill_hall", "regiment", "post_office",       "fire_station", "courthouse", "consulate", "consulate_general",       "Consulate_General_of_The_Bahamas", "Hong_Kong_Economic_and_Trade_Office",       "Government_body_of_Australia",       "National_Historical_Commission_of_the_Philippines_historical_marker",       "municipal_police_of_Canada", "administrative_building",       "building_of_public_administration", "government_agency",       "government_organization", "parliament_building")assign("Culture & Performing Arts",       "opera_company", "opera_house", "performance_hall", "concert_hall", "auditorium",       "rehearsal_room", "conservatory", "cultural_center", "cultural_institution",       "dance_organization", "dance_troupe", "artist-run_space",       "Canadian_artist-run_centre")assign("Libraries & Archives",       "music_library", "media_library", "film_library", "music_archive", "film_archive",       "art_library", "museum_library", "repository", "collection")# --- publications and broadcasters: not physical destinations at all ---assign("Publishing & Media",       "academic_journal", "scientific_journal", "medical_journal",       "open-access_journal", "APC-funded_journal", "open-access_publisher",       "academic_publisher", "publishing_house", "radio_station")assign("Healthcare",       "hospital", "former_hospital", "psychiatric_hospital", "university_hospital",       "academic_medical_centre", "pharmacy", "faculty_of_pharmacy", "morgue")# --- religious schools are schools: children travel to them daily ---assign("Education",       "Catholic_school", "Jesuit_school", "yeshiva", "law_school", "day_care",       "lecture_hall", "student_center", "Teachers'_seminar",       "astronomical_observatory", "university_building", "academic_building",       "school_building")# --- non-Christian places of worship, which clustered with cemeteries ---assign("Religion & Worship",       "synagogue", "Hindu_temple", "Jain_temple", "mosque", "gurdwara",       "Masonic_temple", "BAPS_Shri_Swaminarayan_Mandir_temple", "minor_basilica",       "national_shrine", "presbytery")assign("Memorials & Cemeteries", "triumphal_arch", "walk_of_fame")assign("Museums, Heritage & Attractions",       "fort", "tumulus", "open-air_museum", "tourist_attraction", "hall_of_fame",       "ice_hockey_hall_of_fame", "planetarium")assign("Food, Drink & Hospitality",       "Jewish_delicatessen", "sushi-ya", "kappō", "dining_room")assign("Residential",       "house", "housing_estate", "terrace_of_houses", "rooming_house", "duplex",       "villa", "apartment_building", "condominium", "condominium_complex",       "residential_tower", "tower_block", "log_cabin")# --- organisations with no fixed public destination ---assign("Organisations & Institutions",       "headquarters", "booking_agency", "historical_society", "learned_society",       "labor_union", "trade_association", "association", "Salvation_Army_corps",       "innovation_hub", "division", "architectural_firm")# --- events and incidents: things that happened, not places that exist ---assign("Events & Incidents",       "agricultural_show", "anime_convention", "award", "event", "explosion",       "gas_explosion", "mass_shooting", "student_protest", "train_wreck",       "art_exhibition", "music_festival", "jazz_festival", "benefit_concert",       "sports_competition")assign("Settlement & Districts", "quarter")# --- structures with no clear function attached ---assign("Generic Buildings & Structures",       "mixed-use_development", "stable", "gate", "rotunda", "facility", "edifice",       "pavilion", "changing_room", "office", "ice_house")# --- genuinely unclassifiable. Better an honest bin than a forced category ---assign("Other / Unclassified",       "contaminated_land", "individual_animal", "camp", "signage", "occupation",       "proposed_entity", "IT_support", "managed_services", "service_desk")print(f"tokens with an explicit manual decision: {len(OVERRIDE)}")

In [ ]:
# Apply the overrides on top of the cluster names.tokens["final_category"] = [    OVERRIDE.get(tok, name)    for tok, name in zip(tokens["wikidata_token"], tokens["cluster_name"])]# Flag only the ones where the override actually CHANGED the answer.# Some overrides agree with the cluster; those are not corrections.tokens["manually_corrected"] = tokens["final_category"] != tokens["cluster_name"]n_fixed = tokens["manually_corrected"].sum()print(f"tokens moved out of their MiniLM cluster: {n_fixed} of {len(tokens)} "      f"({100 * n_fixed / len(tokens):.1f}%)")print(f"final categories: {tokens['final_category'].nunique()}")print()print("--- worked example ---")example = ["metro_station", "underground_station", "railway_station", "fire_station",           "coal-fired_power_station", "high_school", "primary_school",           "Catholic_school", "urban_park", "street"]print(tokens[tokens["wikidata_token"].isin(example)]      [["wikidata_token", "minilm_cluster", "cluster_name",        "final_category", "manually_corrected"]]      .to_string(index=False))

---## 9. Apply the mapping back to all rowsThe mapping was built on 538 unique tokens. Now it gets merged back onto all 2,707zone-token rows, and pivoted into a zone × category matrix — the thing you would actuallyfeed to a model.

In [ ]:
categorised = raw.merge(    tokens[["wikidata_token", "minilm_cluster", "final_category"]],    on="wikidata_token", how="left")# If either assert fires, the merge lost or duplicated rows. Stop and fix it.assert len(categorised) == len(raw), "merge changed the row count"assert categorised["final_category"].notna().all(), "some tokens got no category"# Zone x category count matrix: how many entities of each type each zone has.zone_cat = (categorised            .pivot_table(index="loc_id", columns="final_category",                         values="wikidata_token", aggfunc="count")            .fillna(0).astype(int))tokens.to_csv(f"{OUT_DIR}/token_category_mapping.csv", index=False)categorised.to_csv(f"{OUT_DIR}/melted_wikidata_categorised.csv", index=False)zone_cat.to_csv(f"{OUT_DIR}/zone_category_counts.csv")print("zone x category matrix:", zone_cat.shape)print()summary = (categorised.groupby("final_category")           .agg(occurrences=("wikidata_token", "size"),                unique_tokens=("wikidata_token", "nunique"),                zones=("loc_id", "nunique"))           .sort_values("occurrences", ascending=False))summary["pct_of_zones"] = (100 * summary["zones"] / raw["loc_id"].nunique()).round(1)print(summary.to_string())

---## 10. Did this actually help?Do not skip this. Grouping tokens **cannot create information** — it can only tradedetail for density. This section measures what the trade cost.Three checks:1. **Sparsity vs duplication.** Fewer columns means fewer zeros, but also more zones   that look identical to each other.2. **Effective dimensionality.** How many dimensions the matrix really uses, as opposed   to how many columns it has. Computed as the participation ratio of the eigenvalues:   `(Σλ)² / Σλ²`. A matrix with 23 columns but an effective dimension of 2 is carrying   about two numbers' worth of information.3. **Is it just a count?** If the first principal component tracks the raw number of   Wikidata entities per zone, then the "semantic" features are really a density feature   in disguise.

In [ ]:
# How many dimensions the data really occupies (participation ratio).def effective_dim(X):    X = X - X.mean(axis=0)    eigenvalues = np.linalg.svd(X, full_matrices=False)[1] ** 2    return (eigenvalues.sum() ** 2) / (eigenvalues ** 2).sum()# Sparsity, duplicate zones, and effective dimensionality of a zone matrix.def profile(matrix, label):    X = matrix.values.astype(float)    duplicates = pd.DataFrame(np.round(X, 6)).duplicated().sum()    print(f"{label:32s} cols={X.shape[1]:4d}  "          f"zeros={100 * (X == 0).mean():5.1f}%  "          f"duplicate zones={duplicates:4d} ({100 * duplicates / len(X):4.1f}%)  "          f"effective dim={effective_dim(X):5.1f}")# Before: one column per raw token.zone_tok = (categorised            .pivot_table(index="loc_id", columns="wikidata_token",                         values="final_category", aggfunc="count")            .fillna(0))profile(zone_tok, "BEFORE: raw tokens")profile(zone_cat, "AFTER : categories")print()print("Reading it: grouping cuts the zeros, but MORE zones now have an identical")print("profile, and the effective dimensionality drops. That is information lost.")

In [ ]:
# Check 3: are the category counts just a proxy for 'how many POIs are here'?M = zone_cat.values.astype(float)total_entities = M.sum(axis=1)centred = M - M.mean(axis=0)u, s, vt = np.linalg.svd(centred, full_matrices=False)pc1 = u[:, 0]correlation = abs(np.corrcoef(pc1, total_entities)[0, 1])print(f"effective dim of category counts      : {effective_dim(M):.1f} out of {M.shape[1]}")print(f"|corr(1st PC, total entity count)|    : {correlation:.3f}")print()# Now strip density out by converting counts to shares, and see what is left.shares = M / np.clip(M.sum(axis=1, keepdims=True), 1e-9, None)share_dups = pd.DataFrame(np.round(shares, 6)).duplicated().mean()print(f"effective dim of composition (shares) : {effective_dim(shares):.1f}")print(f"duplicate zones under shares          : {100 * share_dups:.1f}%")print()print("If the correlation above is near 1, the counts are mostly POI density.")print("Shares remove density, but with a median of ~3 tokens per zone a share can only")print("be 0, 1/3, 2/3 or 1 — so the extra detail is largely quantisation noise.")

In [ ]:
# Is 24 the right number of groups? Sweep it and look at the trade-off directly.# k = number of unique tokens means "no grouping at all" — the baseline to beat.print(f"{'groups':>7} {'duplicate zones':>17} {'eff dim counts':>16} {'eff dim shares':>16}")for k in [6, 8, 12, 16, 24, 32, 48, len(tokens)]:    if k == len(tokens):        grouping = dict(zip(tokens["wikidata_token"], range(len(tokens))))    else:        lab = AgglomerativeClustering(n_clusters=k, linkage="ward").fit_predict(E)        grouping = dict(zip(tokens["wikidata_token"], lab))    grouped = raw.assign(g=raw["wikidata_token"].map(grouping))    X = (grouped.pivot_table(index="loc_id", columns="g",                             values="wikidata_token", aggfunc="count")                .fillna(0).values.astype(float))    Xs = X / np.clip(X.sum(axis=1, keepdims=True), 1e-9, None)    dup = pd.DataFrame(np.round(X, 6)).duplicated().mean()    tag = " (no grouping)" if k == len(tokens) else ""    print(f"{k:>7} {100 * dup:>16.1f}% {effective_dim(X):>16.1f} "          f"{effective_dim(Xs):>16.1f}{tag}")print()print("The trade-off is monotone: coarser grouping always means more duplicate zones")print("and less information. There is no optimal k — only a choice about how much")print("detail you are willing to give up for interpretability.")

---## 11. What to do with this**Outputs written**| File | Contents ||---|---|| `token_category_mapping.csv` | 538 tokens → cluster → final category, with a correction flag || `melted_wikidata_categorised.csv` | the original 2,707 rows plus a category column || `zone_category_counts.csv` | zone × category matrix, model-ready || `minilm_token_embeddings.npy` | the 538 × 384 embedding matrix |**Before using the categories as features, run these two controls.**1. **Density-only control.** Fit a model with a single feature: `log1p(number of Wikidata   entities in the zone)`. If the full category matrix does not beat it, the semantics   contribute nothing and the categories are a density feature wearing a costume.2. **Split density from composition.** Feed `log1p(count)` and the category *shares* as   separate feature blocks. Otherwise the two are entangled and any gain cannot be   attributed to either one.**For the graph model.** Zones with no Wikidata text should get an all-zero categoryvector plus a `has_wikidata_text` flag. That is more honest than giving them theembedding of an empty string, which makes hundreds of nodes share one arbitrary non-zeroposition in the embedding space.**Wording for the methods section.**> MiniLM embeddings were used to quantify semantic similarity between Wikidata type> labels and to support the construction of broader functional categories. Candidate> clusters were subsequently reviewed against Wikidata semantics and their functional> relevance to the urban environment, with a documented manual correction applied to> tokens whose lexical similarity did not correspond to functional similarity.Report the correction rate. A high rate is a legitimate result about how short typelabels behave under sentence embeddings, not a weakness in the method.